<a href="https://colab.research.google.com/github/Sunidhishree/flyrank-ml-internship1/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Sunidhishree/flyrank-ml-internship1"
REPO_DIR = "flyrank-ml-internship1"
if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.shape)

(30000, 44)


## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [2]:
feat = df.copy()

# Categorical handling: content_type has structural missingness per the data dictionary,
# so use has_-flags instead of blind fillna(0)
feat["has_word_count"] = feat["word_count"].notna().astype(int)
feat["word_count_filled"] = feat["word_count"].fillna(0)

feat["has_engagement_rate"] = feat["engagement_rate"].notna().astype(int)
feat["engagement_rate_filled"] = feat["engagement_rate"].fillna(0)

# avg_position = 0 means "no data" — separate that out rather than treat as a real position
feat["has_position_data"] = (feat["avg_position"] > 0).astype(int)
feat["avg_position_clean"] = np.where(feat["avg_position"] > 0, feat["avg_position"], np.nan)

# Engineered feature: staleness bucket as an ordinal-ish numeric
feat["days_since_last_update_filled"] = feat["days_since_last_update"].fillna(feat["days_since_last_update"].median())

feature_vector = feat[[
    "content_id", "impressions_90d", "ctr", "avg_position_clean", "has_position_data",
    "days_since_last_update_filled", "word_count_filled", "has_word_count",
    "engagement_rate_filled", "has_engagement_rate"
]]
feature_vector.head()

,content_id,impressions_90d,ctr,avg_position_clean,has_position_data,days_since_last_update_filled,word_count_filled,has_word_count,engagement_rate_filled,has_engagement_rate
0,content_304f48230142,3803,0.76,10.6,1,20,3221.0,1,5.88,1
1,content_a1fb4e703a9e,15320,0.05,20.3,1,25,2481.0,1,0.00,1
2,content_9aa793d4d895,12581,0.09,36.5,1,20,3515.0,1,0.00,1
3,content_331d6c4de07b,11751,0.49,6.2,1,22,0.0,0,1.28,1
4,content_d99b7a2d90ca,19140,0.13,44.0,1,14,2803.0,1,0.00,1


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

impressions_90d — trailing 90-day impression count. No missingness observed. Available at decision time: yes, it's a trailing/lagging metric, fully known.
ctr — click-through rate, already a ×100 percentage scale. No missingness. Available at decision time: yes.
avg_position_clean — average search position, with 0 values converted to NaN since 0 means "no rank data," not literal position zero. has_position_data flag added so the model can distinguish "genuinely no data" from "actual low position" instead of silently imputing a misleading value. Available at decision time: yes.
days_since_last_update_filled — days since last content edit; missing values filled with the column median rather than zero, since zero would falsely imply "just updated." Available at decision time: yes.
word_count_filled / has_word_count — page length; per the data dictionary, missingness here correlates with content_type, so a has_ flag preserves that signal instead of a blind fillna injecting a false "0 words" value. Available at decision time: yes.
engagement_rate_filled / has_engagement_rate — same missingness-flag pattern as word_count, for the same reason (content-type-driven missingness, not random).

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [3]:
# Attack: does trend_direction (a label-derived column) sneak correlation into our features?
leak_check = df.copy()
leak_check["is_declining_label"] = leak_check["trend_direction"].str.lower().eq("down").astype(int)

# Test: correlate every candidate feature against the label-derived column
candidate_cols = ["impressions_90d", "ctr", "avg_position", "days_since_last_update", "word_count", "engagement_rate"]
corrs = leak_check[candidate_cols + ["is_declining_label"]].corr(numeric_only=True)["is_declining_label"].drop("is_declining_label")
print("Correlation with label-derived column (should all be modest, not near ±1):")
print(corrs.sort_values(key=abs, ascending=False))

# Explicit leakage trap: trend_pct is the exact number trend_direction is bucketed from
print("\nDirect leakage test — trend_pct vs trend_direction:")
print(leak_check.groupby("trend_direction")["trend_pct"].describe()[["min","max"]])

Correlation with label-derived column (should all be modest, not near ±1):
word_count                0.090157
days_since_last_update    0.081383
ctr                      -0.061911
avg_position             -0.029035
impressions_90d          -0.018175
engagement_rate          -0.012743
Name: is_declining_label, dtype: float64

Direct leakage test — trend_pct vs trend_direction:
                   min      max
trend_direction                
down            -100.0    -20.0
flat               NaN      NaN
new                NaN      NaN
stable           -20.0     20.0
up                20.0  44900.0


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

trend_direction — this is the value my earlier notebooks used to construct a label; using it as a feature would mean training on the answer itself.
trend_pct — the exact percentage trend_direction is derived from; same leakage risk, even more direct.
content_id, client_id — pseudonymous IDs; used only for grouping/joining, never as model features, since they carry no real signal and risk the model latching onto ID patterns instead of genuine behavior.
Any product/decision flags (e.g., "needs CTR fix," health scores) — not present in the starter dataset by design, but noting explicitly that if they appeared in the full warehouse, I would exclude them, since they'd represent someone's already-made decision, not an observable input.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.